In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt

# =========================
# Load FFT filtered data
# =========================
y_fit_AT = np.loadtxt(r"AT.txt")
y_fit_GC = np.loadtxt(r"GC.txt")
y_fit_TCTA = np.loadtxt(r"TCTA.txt")
y_fit_TG = np.loadtxt(r"TG.txt")

# Make all arrays the same length
min_len = min(len(y_fit_AT), len(y_fit_GC), len(y_fit_TCTA), len(y_fit_TG))

y_fit_AT = y_fit_AT[:min_len]
y_fit_GC = y_fit_GC[:min_len]
y_fit_TCTA = y_fit_TCTA[:min_len]
y_fit_TG = y_fit_TG[:min_len]

# Time axis for 5.5 min data
x_fit = np.linspace(0, 5.5, min_len)

spline_dict = {
    "AT": interp1d(x_fit, y_fit_AT, kind='cubic', fill_value="extrapolate"),
    "GC": interp1d(x_fit, y_fit_GC, kind='cubic', fill_value="extrapolate"),
    "TCTA": interp1d(x_fit, y_fit_TCTA, kind='cubic', fill_value="extrapolate"),
    "TG": interp1d(x_fit, y_fit_TG, kind='cubic', fill_value="extrapolate"),
}

def hill(x, A, B, C, N):
    x = np.clip(x, 1e-6, 1e6)
    return A * (x**N) / (B**N + x**N) + C

H2O2_params = {
    "AT": [0.936384942, 0.001085287, 0.020947926, 0.893406228],
    "GC": [0.793063901, 0.003510244, 0.07686459, 0.936160936],
    "TCTA": [0.773519649, 0.001507455, 0.155108873, 0.557479242],
    "TG": [0.855612178, 0.000794612, 0.070094087, 0.758826963],
}
OH_params = {
    "AT": [0.916990085, 0.000253656, 0.069155176, 1.732128602],
    "GC": [0.991283169, 0.000319827, 1.20754e-21, 1.866347581],
    "TCTA": [0.947735022, 2.65811e-05, 3.49404e-21, 1.539953359],
    "TG": [0.855692132, 5.37807e-06, 0.087536377, 2.431551544],
}
NO3_params = {
    "AT": [0.673375337, 0.00502307, 1.62487e-17, 13.57349854],
    "GC": [0.913292006, 0.003669045, 1.94032e-23, 1.661215552],
    "TCTA": [0.65500641, 1.071487052, 1.10953e-17, 7.309164656],
    "TG": [1.128907612, 0.004846066, 2.51132e-21, 1.20922578],
}
NO2_params = {
    "AT": [0.854516206, 6.62637e-07, 2.23838e-23, 0.649132913],
    "GC": [0.849766466, 6.80344e-06, 2.26173e-19, 2.155228561],
    "TCTA": [0.910793206, 1.96472e-05, 2.31943e-21, 1.349954466],
    "TG": [0.644761024, 1.46385e-06, 0.223077273, 1.768813238],
}

param_sets = [H2O2_params, OH_params, NO3_params, NO2_params]

# =========================
# Chemometric decoding
# =========================
x_time = np.linspace(0, 5.5, min_len)

x0 = [0.001, 0.001, 0.01, 0.001]
x1_all, x2_all, x3_all, x4_all = [], [], [], []

for t in x_time:
    R_target = [spline_dict[seq](t) for seq in ["AT", "GC", "TCTA", "TG"]]

    def objective(x):
        x_sum = np.sum(x)
        if x_sum <= 0:
            return 1e9

        error = 0
        for j, seq in enumerate(["AT", "GC", "TCTA", "TG"]):
            Ri_pred = 0
            for k in range(4):
                A, B, C, N = param_sets[k][seq]
                Ri_pred += (x[k] / x_sum) * hill(x[k], A, B, C, N)
            error += (Ri_pred - R_target[j]) ** 2
        return error

    bounds = [(0.0001, 0.1), (0, None), (0.00001, 0.1), (0, None)]
    result = minimize(objective, x0=x0, bounds=bounds)

    sol = result.x
    x0 = sol

    x1_all.append(sol[0])
    x2_all.append(sol[1])
    x3_all.append(sol[2])
    x4_all.append(sol[3])

# =========================
# Plot
# =========================
fig, axs = plt.subplots(4, 1, figsize=(8, 8), sharex=True)

axs[0].plot(x_time, x1_all, color='r')
axs[0].set_ylabel('H2O2')

axs[1].plot(x_time, x2_all, color='g')
axs[1].set_ylabel('OH')

axs[2].plot(x_time, x3_all, color='b')
axs[2].set_ylabel('NO3')

axs[3].plot(x_time, x4_all, color='purple')
axs[3].set_ylabel('NO2')
axs[3].set_xlabel('Time [min]')

plt.tight_layout()
plt.show()